# Boundary quality assurance

Generic checks for reporting-boundary layers before spatial summarisation.


In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone

import geopandas as gpd
import pandas as pd

HERE = Path.cwd()
ROOT = HERE.parent if HERE.name == 'notebooks' else HERE
DATA = ROOT / 'data_raw'
OUTPUTS = ROOT / 'outputs'
TABLES = ROOT / 'tables'
for folder in (OUTPUTS, TABLES):
    folder.mkdir(parents=True, exist_ok=True)

BOUNDARY_FILE = DATA / 'boundary.gpkg'
ID_FIELD = None
EQUAL_AREA_CRS = 'EPSG:6933'

if not BOUNDARY_FILE.exists():
    raise FileNotFoundError(f'Boundary file not found: {BOUNDARY_FILE}')

gdf = gpd.read_file(BOUNDARY_FILE)
checks = [
    {'check': 'feature_count', 'status': 'PASS' if len(gdf) > 0 else 'FAIL', 'value': int(len(gdf))},
    {'check': 'crs_defined', 'status': 'PASS' if gdf.crs is not None else 'FAIL', 'value': str(gdf.crs)},
    {'check': 'missing_geometry', 'status': 'PASS' if int(gdf.geometry.isna().sum()) == 0 else 'FAIL', 'value': int(gdf.geometry.isna().sum())},
    {'check': 'empty_geometry', 'status': 'PASS' if int(gdf.geometry.is_empty.sum()) == 0 else 'FAIL', 'value': int(gdf.geometry.is_empty.sum())},
    {'check': 'invalid_geometry', 'status': 'PASS' if int((~gdf.geometry.is_valid).sum()) == 0 else 'FAIL', 'value': int((~gdf.geometry.is_valid).sum())},
]

geom_hash = gdf.geometry.apply(lambda geom: geom.wkb_hex if geom is not None else None)
duplicated_geom = gdf[geom_hash.duplicated(keep=False)].copy()
checks.append({'check': 'duplicate_geometries', 'status': 'PASS' if duplicated_geom.empty else 'WARN', 'value': int(len(duplicated_geom))})

if ID_FIELD and ID_FIELD in gdf.columns:
    duplicated_ids = gdf[gdf[ID_FIELD].duplicated(keep=False)].copy()
    checks.append({'check': 'duplicate_ids', 'status': 'PASS' if duplicated_ids.empty else 'WARN', 'value': int(len(duplicated_ids))})

if gdf.crs is not None:
    gdf_area = gdf.to_crs(EQUAL_AREA_CRS)
    gdf['area_km2'] = gdf_area.geometry.area / 1_000_000
    gdf.drop(columns='geometry').to_csv(TABLES / 'boundary_area_diagnostics.csv', index=False)

qa = pd.DataFrame(checks)
qa.to_csv(TABLES / 'boundary_qa_summary.csv', index=False)
with open(OUTPUTS / 'boundary_qa_summary.json', 'w', encoding='utf-8') as f:
    json.dump({'run_time_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'), 'checks': checks}, f, indent=2)
qa
